# ============================================
# MODULE 1: FEATURE ENGINEERING FOR ENERGY SYSTEMS
# ============================================
#
# Learning Objectives:
# - Create meaningful features from raw power systems data
# - Engineer time-based features for temporal patterns
# - Calculate electrical domain-specific features
# - Perform feature scaling and normalization
# - Apply feature selection techniques
# - Handle categorical variables in power systems
#
# Real-World Application:
# In power systems analysis, raw sensor data often needs transformation into
# meaningful features. Feature engineering is the process of creating new variables
# that better represent the underlying patterns in your data. Good features can:
# - Improve model accuracy by 10-50% or more
# - Reduce training time significantly
# - Make models more interpretable for stakeholders
# - Capture domain knowledge about electrical systems
#
# Estimated Time: 4-5 hours
# ============================================

## Section 1: Import Libraries and Setup

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib for visualizations
import matplotlib.pyplot as plt

# Import seaborn for enhanced statistical plots
import seaborn as sns

# Import datetime for time-based operations
from datetime import datetime, timedelta

# Import sklearn preprocessing tools
# StandardScaler: standardizes features by removing mean and scaling to unit variance
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Import OneHotEncoder for categorical variables
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

# Import feature selection methods
# SelectKBest: selects top k features based on statistical tests
from sklearn.feature_selection import SelectKBest, f_regression, mutual_info_regression

# Import variance threshold for removing low-variance features
from sklearn.feature_selection import VarianceThreshold

# Import PolynomialFeatures for creating interaction terms
from sklearn.preprocessing import PolynomialFeatures

# Import scipy for statistical operations
from scipy import stats
from scipy.fft import fft, fftfreq

# Import warnings to suppress unnecessary messages
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## Section 2: Generate Comprehensive Power System Dataset

We'll create a realistic dataset with multiple features for feature engineering practice.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Generate 3 months of hourly data for comprehensive analysis
n_records = 2160  # 90 days × 24 hours

# Create timestamp range starting from January 1, 2023
date_range = pd.date_range(start='2023-01-01', periods=n_records, freq='H')

# Extract temporal components for pattern generation
hours = date_range.hour
day_of_week = date_range.dayofweek
day_of_year = date_range.dayofyear

# Generate realistic load pattern with multiple components
# Component 1: Daily pattern (24-hour cycle)
daily_pattern = 100 + 50 * np.sin((hours - 6) * np.pi / 12)

# Component 2: Weekly pattern (weekday vs weekend)
weekly_pattern = np.where(day_of_week < 5, 1.0, 0.85)

# Component 3: Seasonal pattern (winter higher due to heating)
# Using cosine to create winter peak
seasonal_pattern = 1.0 + 0.15 * np.cos((day_of_year - 15) * 2 * np.pi / 365)

# Combine all patterns to create realistic load
base_load = daily_pattern * weekly_pattern * seasonal_pattern

# Add random noise to simulate real-world variability
load_mw = base_load + np.random.normal(0, 8, n_records)

# Generate voltage with load-dependent variation
# Voltage drops slightly under heavy load (voltage regulation)
voltage_kv = 230 - (load_mw - load_mw.mean()) * 0.015 + np.random.normal(0, 1.5, n_records)

# Generate current based on power and voltage
# Using three-phase power relationship: P = √3 × V × I × PF
# Solving for I: I = P / (√3 × V × PF)
power_factor = np.random.uniform(0.88, 0.96, n_records)
current_a = (load_mw * 1000) / (np.sqrt(3) * voltage_kv * power_factor)

# Add measurement noise to current
current_a = current_a + np.random.normal(0, 15, n_records)

# Generate frequency with small variations
# Frequency deviates based on generation-load balance
frequency_hz = 60.0 + (load_mw - load_mw.mean()) * 0.0001 + np.random.normal(0, 0.015, n_records)

# Calculate reactive power from real power and power factor
# Q = P × tan(arccos(PF))
reactive_power_mvar = load_mw * np.tan(np.arccos(power_factor))

# Calculate apparent power
# S = √(P² + Q²) or S = P / PF
apparent_power_mva = load_mw / power_factor

# Generate weather-related features
# Temperature varies with season and time of day
daily_temp_var = 5 * np.sin((hours - 14) * np.pi / 12)  # Peak at 2 PM
seasonal_temp = 15 + 12 * np.sin((day_of_year - 80) * 2 * np.pi / 365)  # Peak in summer
temperature_c = seasonal_temp + daily_temp_var + np.random.normal(0, 2, n_records)

# Wind speed affects renewable generation
# Higher in winter and varies throughout day
wind_speed_ms = 8 + 4 * np.sin((day_of_year - 15) * 2 * np.pi / 365) + np.random.exponential(3, n_records)
wind_speed_ms = np.clip(wind_speed_ms, 0, 25)  # Physical limit for wind speed

# Solar irradiance (only during daylight hours)
# Zero at night, peak around noon
solar_irradiance = np.where(
    (hours >= 6) & (hours <= 18),
    1000 * np.sin((hours - 6) * np.pi / 12) * (0.8 + 0.2 * np.random.random(n_records)),
    0
)

# Humidity affects electrical equipment performance
humidity_percent = np.random.uniform(35, 85, n_records)

# Generate transformer load as percentage of rated capacity
# Related to system load
transformer_load_percent = (load_mw / load_mw.max()) * 100

# Generate oil temperature for transformer (monitoring critical parameter)
# Depends on load and ambient temperature
oil_temperature_c = 40 + transformer_load_percent * 0.3 + temperature_c * 0.2 + np.random.normal(0, 3, n_records)

# Create the comprehensive DataFrame
df = pd.DataFrame({
    'timestamp': date_range,
    'load_mw': load_mw,
    'voltage_kv': voltage_kv,
    'current_a': current_a,
    'frequency_hz': frequency_hz,
    'power_factor': power_factor,
    'reactive_power_mvar': reactive_power_mvar,
    'apparent_power_mva': apparent_power_mva,
    'temperature_c': temperature_c,
    'wind_speed_ms': wind_speed_ms,
    'solar_irradiance': solar_irradiance,
    'humidity_percent': humidity_percent,
    'transformer_load_percent': transformer_load_percent,
    'oil_temperature_c': oil_temperature_c
})

print(f"Generated {len(df)} power system records")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Total features: {len(df.columns)}")
print("\nFirst 5 rows:")
print(df.head())

## Section 3: Temporal Feature Engineering

Extract time-based features that capture patterns in power systems data.

In [ ]:
# Extract basic temporal features from timestamp
# These features help capture daily, weekly, and seasonal patterns

# Hour of day (0-23): captures daily load curves
df['hour'] = df['timestamp'].dt.hour

# Day of week (0=Monday, 6=Sunday): captures weekly patterns
df['day_of_week'] = df['timestamp'].dt.dayofweek

# Day of month (1-31): can capture billing cycle patterns
df['day_of_month'] = df['timestamp'].dt.day

# Month (1-12): captures seasonal patterns
df['month'] = df['timestamp'].dt.month

# Day of year (1-365): captures annual cycles
df['day_of_year'] = df['timestamp'].dt.dayofyear

# Week of year (1-52): useful for weekly aggregations
df['week_of_year'] = df['timestamp'].dt.isocalendar().week

# Quarter (1-4): seasonal business patterns
df['quarter'] = df['timestamp'].dt.quarter

print("Basic temporal features created:")
print(df[['timestamp', 'hour', 'day_of_week', 'month', 'day_of_year']].head(10))

In [ ]:
# Create cyclical features using sine and cosine transformations
# Cyclical encoding is crucial for time features because:
# - Hour 23 and Hour 0 are actually close in time
# - Month 12 and Month 1 are consecutive
# - Linear encoding (0, 1, 2, ..., 23) doesn't capture this circular nature

# Encode hour as cyclical feature (24-hour cycle)
# sin and cos create a 2D representation of the circle
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

# Encode day of week as cyclical feature (7-day cycle)
df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Encode month as cyclical feature (12-month cycle)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# Encode day of year as cyclical feature (365-day cycle)
df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

print("Cyclical temporal features created")
print("\nExample: Hour encoding (linear vs cyclical)")
print(df[['hour', 'hour_sin', 'hour_cos']].head(25))

In [ ]:
# Visualize cyclical encoding to understand the concept
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Circular representation of hours
# This shows how cyclical encoding maps time to a circle
unique_hours = np.arange(24)
hour_sin = np.sin(2 * np.pi * unique_hours / 24)
hour_cos = np.cos(2 * np.pi * unique_hours / 24)

axes[0].scatter(hour_sin, hour_cos, c=unique_hours, cmap='twilight', s=200, edgecolor='black', linewidth=2)
for i, hour in enumerate(unique_hours):
    axes[0].annotate(f'{hour}:00', (hour_sin[i], hour_cos[i]), 
                     textcoords="offset points", xytext=(0,10), ha='center', fontweight='bold')
axes[0].set_xlabel('Hour Sin', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Hour Cos', fontsize=12, fontweight='bold')
axes[0].set_title('Cyclical Encoding of Hours (24-hour cycle)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

# Add circle to show the unit circle
circle = plt.Circle((0, 0), 1, fill=False, color='red', linestyle='--', linewidth=2)
axes[0].add_patch(circle)

# Plot 2: Time series of sin and cos over 48 hours
hours_48 = np.arange(48)
hour_sin_48 = np.sin(2 * np.pi * (hours_48 % 24) / 24)
hour_cos_48 = np.cos(2 * np.pi * (hours_48 % 24) / 24)

axes[1].plot(hours_48, hour_sin_48, linewidth=2, marker='o', label='Hour Sin', color='blue')
axes[1].plot(hours_48, hour_cos_48, linewidth=2, marker='s', label='Hour Cos', color='red')
axes[1].axvline(x=24, color='green', linestyle='--', linewidth=2, label='Day Boundary')
axes[1].set_xlabel('Hour', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=12, fontweight='bold')
axes[1].set_title('Cyclical Features Over 48 Hours', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

print("Note: Cyclical encoding ensures hour 23 and hour 0 are close in feature space")

In [ ]:
# Create binary temporal features for specific conditions
# These are useful for capturing threshold-based behaviors

# Is weekend flag (Saturday=5, Sunday=6)
# Industrial loads typically drop on weekends
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# Is business hours (8 AM to 6 PM on weekdays)
# Commercial loads peak during business hours
df['is_business_hours'] = ((df['hour'] >= 8) & (df['hour'] < 18) & (df['day_of_week'] < 5)).astype(int)

# Is peak hours (typically 4 PM to 9 PM when demand is highest)
# Used for demand response and peak pricing
df['is_peak_hours'] = ((df['hour'] >= 16) & (df['hour'] < 21)).astype(int)

# Is night (10 PM to 6 AM - minimum load period)
df['is_night'] = ((df['hour'] >= 22) | (df['hour'] < 6)).astype(int)

# Is summer (June, July, August - cooling load dominant)
df['is_summer'] = df['month'].isin([6, 7, 8]).astype(int)

# Is winter (December, January, February - heating load dominant)
df['is_winter'] = df['month'].isin([12, 1, 2]).astype(int)

print("Binary temporal features created:")
print(df[['timestamp', 'is_weekend', 'is_business_hours', 'is_peak_hours', 'is_night']].head(20))

## Section 4: Lag Features (Historical Values)

Create features based on past values - crucial for time series forecasting.

In [ ]:
# Create lag features for load (past values)
# Lag features capture temporal dependencies:
# - Load at hour t often depends on load at hour t-1, t-24, etc.
# - These are essential for time series forecasting

# Lag 1 hour (load 1 hour ago)
# Most recent past value, strong predictor
df['load_lag_1h'] = df['load_mw'].shift(1)

# Lag 24 hours (load at same time yesterday)
# Captures daily cycle - same hour yesterday is often similar
df['load_lag_24h'] = df['load_mw'].shift(24)

# Lag 168 hours (load at same time last week)
# Captures weekly patterns - same day/hour last week
df['load_lag_168h'] = df['load_mw'].shift(168)

# Create multiple short-term lags (2, 3, 6, 12 hours)
for lag in [2, 3, 6, 12]:
    df[f'load_lag_{lag}h'] = df['load_mw'].shift(lag)

print("Lag features created for load")
print("\nExample: Current load vs lagged values")
print(df[['timestamp', 'load_mw', 'load_lag_1h', 'load_lag_24h', 'load_lag_168h']].iloc[168:178])

In [ ]:
# Create rolling window features (moving statistics)
# Rolling windows capture trends and variability over time

# Rolling mean over last 24 hours
# Smooths out short-term fluctuations, shows daily trend
df['load_rolling_mean_24h'] = df['load_mw'].rolling(window=24, min_periods=1).mean()

# Rolling standard deviation over last 24 hours
# Measures recent volatility in load
df['load_rolling_std_24h'] = df['load_mw'].rolling(window=24, min_periods=1).std()

# Rolling minimum over last 24 hours
df['load_rolling_min_24h'] = df['load_mw'].rolling(window=24, min_periods=1).min()

# Rolling maximum over last 24 hours
df['load_rolling_max_24h'] = df['load_mw'].rolling(window=24, min_periods=1).max()

# Rolling mean over last week (168 hours)
# Captures weekly trend
df['load_rolling_mean_168h'] = df['load_mw'].rolling(window=168, min_periods=1).mean()

print("Rolling window features created")
print("\nExample: Load with rolling statistics")
print(df[['timestamp', 'load_mw', 'load_rolling_mean_24h', 'load_rolling_std_24h', 
          'load_rolling_min_24h', 'load_rolling_max_24h']].iloc[48:58])

In [ ]:
# Create rate of change features
# These capture how quickly load is changing (important for grid stability)

# Hour-over-hour change (absolute)
# Difference between current and previous hour
df['load_change_1h'] = df['load_mw'].diff(1)

# Hour-over-hour change (percentage)
# Relative change is often more meaningful than absolute
df['load_pct_change_1h'] = df['load_mw'].pct_change(1) * 100

# Day-over-day change (same hour yesterday)
df['load_change_24h'] = df['load_mw'].diff(24)

# Week-over-week change
df['load_change_168h'] = df['load_mw'].diff(168)

# Rate of change acceleration (second derivative)
# Measures how the rate of change is changing
df['load_change_acceleration'] = df['load_change_1h'].diff(1)

print("Rate of change features created")
print("\nExample: Load changes")
print(df[['timestamp', 'load_mw', 'load_change_1h', 'load_pct_change_1h']].iloc[24:34])

In [ ]:
# Visualize lag features and rolling statistics
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Select one week of data for visualization
week_data = df.iloc[500:668].copy()  # 168 hours = 1 week

# Plot 1: Current load vs lag features
axes[0].plot(week_data['timestamp'], week_data['load_mw'], 
             linewidth=2, label='Current Load', color='blue', marker='o', markersize=3)
axes[0].plot(week_data['timestamp'], week_data['load_lag_24h'], 
             linewidth=2, label='Load 24h Ago', color='red', linestyle='--', marker='s', markersize=3)
axes[0].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[0].set_title('Current Load vs 24-Hour Lag', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Load with rolling mean and confidence bands
axes[1].plot(week_data['timestamp'], week_data['load_mw'], 
             linewidth=1.5, label='Actual Load', color='blue', alpha=0.6)
axes[1].plot(week_data['timestamp'], week_data['load_rolling_mean_24h'], 
             linewidth=3, label='24h Rolling Mean', color='red')

# Add confidence bands (rolling mean ± rolling std)
upper_band = week_data['load_rolling_mean_24h'] + week_data['load_rolling_std_24h']
lower_band = week_data['load_rolling_mean_24h'] - week_data['load_rolling_std_24h']
axes[1].fill_between(week_data['timestamp'], lower_band, upper_band, 
                      alpha=0.3, color='red', label='±1 Std Dev')

axes[1].set_ylabel('Load (MW)', fontsize=11, fontweight='bold')
axes[1].set_title('Load with Rolling Statistics', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Plot 3: Rate of change
axes[2].plot(week_data['timestamp'], week_data['load_change_1h'], 
             linewidth=2, label='Hour-over-Hour Change', color='green', marker='o', markersize=3)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=2, label='Zero Line')
axes[2].fill_between(week_data['timestamp'], 0, week_data['load_change_1h'], 
                      where=(week_data['load_change_1h'] > 0), alpha=0.3, color='green', label='Increasing')
axes[2].fill_between(week_data['timestamp'], 0, week_data['load_change_1h'], 
                      where=(week_data['load_change_1h'] <= 0), alpha=0.3, color='red', label='Decreasing')
axes[2].set_xlabel('Time', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Load Change (MW)', fontsize=11, fontweight='bold')
axes[2].set_title('Hour-over-Hour Load Change (Ramp Rate)', fontsize=13, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Visualization complete")

## Section 5: Domain-Specific Electrical Features

Create features based on electrical engineering principles and formulas.

In [ ]:
# Calculate additional electrical parameters from raw measurements

# Total Harmonic Distortion (THD) estimation
# THD measures power quality - higher values indicate more distortion
# In real systems, THD is measured; here we estimate based on load variability
df['estimated_thd_percent'] = (df['load_mw'] / df['load_mw'].max()) * 5 + np.random.uniform(0, 2, len(df))

# Voltage deviation from nominal
# Measures how far voltage strays from ideal 230 kV
nominal_voltage = 230
df['voltage_deviation_kv'] = df['voltage_kv'] - nominal_voltage

# Voltage deviation percentage
# Standardized measure for comparing across voltage levels
df['voltage_deviation_percent'] = (df['voltage_deviation_kv'] / nominal_voltage) * 100

# Frequency deviation from nominal
# Critical for grid stability monitoring
nominal_frequency = 60.0
df['frequency_deviation_hz'] = df['frequency_hz'] - nominal_frequency

# Frequency deviation in mHz (more common unit for small deviations)
df['frequency_deviation_mhz'] = df['frequency_deviation_hz'] * 1000

print("Basic electrical features created")
print(df[['voltage_kv', 'voltage_deviation_kv', 'voltage_deviation_percent', 
          'frequency_hz', 'frequency_deviation_mhz']].describe())

In [ ]:
# Calculate power-related features

# Reactive to Active Power Ratio
# Indicates power quality and efficiency
# Higher ratio means lower power factor and more reactive power
df['reactive_to_active_ratio'] = df['reactive_power_mvar'] / df['load_mw']

# Total Power Factor Displacement (DPF)
# Already have power_factor, but let's calculate from first principles
# PF = P / S = cos(φ)
df['power_factor_calculated'] = df['load_mw'] / df['apparent_power_mva']

# Power Factor Angle (phi) in degrees
# Angle between voltage and current phasors
df['power_factor_angle_deg'] = np.arccos(np.clip(df['power_factor'], 0, 1)) * 180 / np.pi

# Load factor (ratio of average to peak load over a period)
# Using 24-hour rolling window
df['load_factor_24h'] = df['load_rolling_mean_24h'] / df['load_rolling_max_24h']

# Demand factor (ratio of maximum demand to connected load)
# Simulated assuming connected capacity is 150% of peak load
connected_capacity = df['load_mw'].max() * 1.5
df['demand_factor'] = df['load_rolling_max_24h'] / connected_capacity

print("Power-related features created")
print(df[['load_mw', 'reactive_power_mvar', 'reactive_to_active_ratio', 
          'power_factor', 'power_factor_angle_deg']].describe())

In [ ]:
# Calculate impedance and resistance features

# Apparent impedance (Z = V / I)
# Ohm's law for AC circuits
# Note: This is simplified; real impedance includes reactance
df['apparent_impedance_ohm'] = (df['voltage_kv'] * 1000) / df['current_a']

# Active power per unit current
# Indicates efficiency of power delivery
df['power_per_ampere'] = df['load_mw'] / (df['current_a'] / 1000)  # MW per kA

# Voltage-Current product (related to apparent power)
df['voltage_current_product'] = df['voltage_kv'] * df['current_a']

print("Impedance features created")
print(df[['voltage_kv', 'current_a', 'apparent_impedance_ohm', 'power_per_ampere']].describe())

In [ ]:
# Calculate transformer-specific features

# Transformer loading margin (% capacity remaining)
df['transformer_loading_margin'] = 100 - df['transformer_load_percent']

# Oil temperature rise above ambient
# Indicates transformer heating due to load losses
df['oil_temp_rise'] = df['oil_temperature_c'] - df['temperature_c']

# Temperature per unit load (thermal efficiency indicator)
# Lower values indicate better cooling
df['temp_per_unit_load'] = df['oil_temp_rise'] / (df['transformer_load_percent'] + 1)  # +1 to avoid division by zero

# Overload flag (transformer loaded above 90%)
df['is_transformer_overload'] = (df['transformer_load_percent'] > 90).astype(int)

# High temperature flag (oil temp above 80°C)
df['is_high_oil_temp'] = (df['oil_temperature_c'] > 80).astype(int)

print("Transformer features created")
print(df[['transformer_load_percent', 'oil_temperature_c', 'oil_temp_rise', 
          'is_transformer_overload', 'is_high_oil_temp']].describe())

In [ ]:
# Calculate renewable energy related features

# Wind power potential (using simplified wind power formula)
# P = 0.5 × ρ × A × v³ × Cp
# Simplified: assume proportional to v³
df['wind_power_potential'] = (df['wind_speed_ms'] ** 3) * 0.01  # Scaled for visualization

# Solar capacity factor (ratio of actual to potential at peak)
# Higher values during peak sun hours
peak_solar = 1000  # W/m²
df['solar_capacity_factor'] = df['solar_irradiance'] / peak_solar

# Combined renewable potential
# Normalized sum of wind and solar
df['renewable_potential'] = (df['wind_power_potential'] / df['wind_power_potential'].max() + 
                             df['solar_capacity_factor']) / 2

# Is high renewable period (when combined potential > 0.6)
df['is_high_renewable'] = (df['renewable_potential'] > 0.6).astype(int)

print("Renewable energy features created")
print(df[['wind_speed_ms', 'wind_power_potential', 'solar_irradiance', 
          'solar_capacity_factor', 'renewable_potential']].describe())

## Section 6: Interaction Features

Create features from combinations of existing features.

In [ ]:
# Create interaction features between weather and load
# Interactions capture non-linear relationships

# Temperature-Load interaction
# Load response to temperature differs at different times
df['temp_load_interaction'] = df['temperature_c'] * df['load_mw']

# Temperature squared (captures non-linear heating/cooling response)
# Load often increases with both high and low temperatures (U-shaped curve)
df['temperature_squared'] = df['temperature_c'] ** 2

# Wind-Solar interaction
# Combined effect of wind and solar generation
df['wind_solar_interaction'] = df['wind_speed_ms'] * df['solar_irradiance']

# Hour-Temperature interaction
# Temperature effect varies by time of day
df['hour_temp_interaction'] = df['hour'] * df['temperature_c']

# Weekend-Hour interaction
# Load pattern differs between weekend and weekday hours
df['weekend_hour_interaction'] = df['is_weekend'] * df['hour']

print("Interaction features created")
print(df[['temperature_c', 'load_mw', 'temp_load_interaction', 'temperature_squared']].head(10))

In [ ]:
# Create polynomial features for key variables
# Polynomial features capture non-linear relationships

# Select key features for polynomial expansion
key_features = ['temperature_c', 'wind_speed_ms', 'humidity_percent']

# Create polynomial features (degree 2: x, x²)
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[key_features])

# Get feature names
poly_feature_names = poly.get_feature_names_out(key_features)

# Add polynomial features to dataframe
# Only add the new features (squared and interaction terms)
# Skip the first 3 which are the original features
for i, name in enumerate(poly_feature_names[3:], start=3):
    # Create readable column names
    clean_name = name.replace(' ', '_').replace('^', '_pow')
    df[f'poly_{clean_name}'] = poly_features[:, i]

print("Polynomial features created")
print(f"New polynomial features: {list(poly_feature_names[3:])}")
print(f"\nTotal features now: {len(df.columns)}")

## Section 7: Aggregation Features

Create statistical aggregations over different time windows.

In [ ]:
# Create expanding window features (cumulative statistics)
# Expanding windows include all data from start to current point

# Cumulative average load
# Historical mean load up to this point
df['load_cumulative_mean'] = df['load_mw'].expanding(min_periods=1).mean()

# Cumulative maximum load
# Peak load observed so far
df['load_cumulative_max'] = df['load_mw'].expanding(min_periods=1).max()

# Cumulative minimum load
df['load_cumulative_min'] = df['load_mw'].expanding(min_periods=1).min()

# Load as percentage of historical maximum
df['load_pct_of_hist_max'] = (df['load_mw'] / df['load_cumulative_max']) * 100

print("Expanding window features created")
print(df[['load_mw', 'load_cumulative_mean', 'load_cumulative_max', 'load_pct_of_hist_max']].head(50))

In [ ]:
# Create exponentially weighted moving averages (EWMA)
# EWMA gives more weight to recent observations
# Useful for capturing trends while being responsive to changes

# EWMA with span=12 hours (half-life of ~8 hours)
df['load_ewma_12h'] = df['load_mw'].ewm(span=12, adjust=False).mean()

# EWMA with span=24 hours (half-life of ~16 hours)
df['load_ewma_24h'] = df['load_mw'].ewm(span=24, adjust=False).mean()

# EWMA standard deviation
df['load_ewma_std_24h'] = df['load_mw'].ewm(span=24, adjust=False).std()

print("Exponential weighted features created")
print(df[['load_mw', 'load_ewma_12h', 'load_ewma_24h', 'load_ewma_std_24h']].iloc[24:34])

In [ ]:
# Compare different smoothing techniques
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Select 2 weeks of data
plot_data = df.iloc[500:836].copy()  # 336 hours = 2 weeks

# Plot 1: Compare rolling mean vs EWMA
axes[0].plot(plot_data['timestamp'], plot_data['load_mw'], 
             linewidth=1, label='Actual Load', color='gray', alpha=0.5)
axes[0].plot(plot_data['timestamp'], plot_data['load_rolling_mean_24h'], 
             linewidth=2.5, label='Rolling Mean (24h)', color='blue')
axes[0].plot(plot_data['timestamp'], plot_data['load_ewma_24h'], 
             linewidth=2.5, label='EWMA (24h)', color='red', linestyle='--')
axes[0].set_ylabel('Load (MW)', fontsize=12, fontweight='bold')
axes[0].set_title('Comparison: Rolling Mean vs EWMA', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Variability measures
axes[1].plot(plot_data['timestamp'], plot_data['load_rolling_std_24h'], 
             linewidth=2, label='Rolling Std (24h)', color='blue', marker='o', markersize=3)
axes[1].plot(plot_data['timestamp'], plot_data['load_ewma_std_24h'], 
             linewidth=2, label='EWMA Std (24h)', color='red', marker='s', markersize=3)
axes[1].set_xlabel('Time', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Standard Deviation (MW)', fontsize=12, fontweight='bold')
axes[1].set_title('Comparison: Load Variability Measures', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: EWMA is more responsive to recent changes than rolling mean")

## Section 8: Feature Scaling and Normalization

Scale features to comparable ranges for machine learning algorithms.

In [ ]:
# Select numerical features for scaling demonstration
# Exclude timestamp and binary features
features_to_scale = ['load_mw', 'voltage_kv', 'current_a', 'frequency_hz', 
                     'temperature_c', 'wind_speed_ms', 'humidity_percent']

# Create a subset DataFrame with selected features (drop NaN values from lag features)
df_for_scaling = df[features_to_scale].iloc[200:].copy()  # Skip first 200 rows with NaN lag features

print("Original feature statistics:")
print(df_for_scaling.describe())
print("\nNote the different scales: voltage is ~230, frequency is ~60, humidity is 0-100")

In [ ]:
# Method 1: StandardScaler (Z-score normalization)
# Transforms features to have mean=0 and std=1
# Formula: x_scaled = (x - mean) / std
# Best for: Features with Gaussian distribution, when outliers are not a concern

scaler_standard = StandardScaler()
df_standard_scaled = scaler_standard.fit_transform(df_for_scaling)
df_standard_scaled = pd.DataFrame(df_standard_scaled, columns=features_to_scale)

print("StandardScaler Results:")
print(df_standard_scaled.describe())
print("\nNote: All features now have mean ≈ 0 and std ≈ 1")

In [ ]:
# Method 2: MinMaxScaler
# Scales features to a fixed range [0, 1]
# Formula: x_scaled = (x - min) / (max - min)
# Best for: When you need bounded values, neural networks, image processing

scaler_minmax = MinMaxScaler()
df_minmax_scaled = scaler_minmax.fit_transform(df_for_scaling)
df_minmax_scaled = pd.DataFrame(df_minmax_scaled, columns=features_to_scale)

print("MinMaxScaler Results:")
print(df_minmax_scaled.describe())
print("\nNote: All features now range from 0 to 1")

In [ ]:
# Method 3: RobustScaler
# Uses median and IQR instead of mean and std
# Formula: x_scaled = (x - median) / IQR
# Best for: Data with outliers (resistant to extreme values)

scaler_robust = RobustScaler()
df_robust_scaled = scaler_robust.fit_transform(df_for_scaling)
df_robust_scaled = pd.DataFrame(df_robust_scaled, columns=features_to_scale)

print("RobustScaler Results:")
print(df_robust_scaled.describe())
print("\nNote: Median ≈ 0, less affected by outliers than StandardScaler")

In [ ]:
# Visualize the effect of different scaling methods
fig, axes = plt.subplots(4, 2, figsize=(16, 16))
fig.suptitle('Comparison of Scaling Methods', fontsize=16, fontweight='bold')

# Select two features to visualize: load_mw and voltage_kv
feature1, feature2 = 'load_mw', 'voltage_kv'

# Original data - Histogram
axes[0, 0].hist(df_for_scaling[feature1], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[0, 0].set_title(f'{feature1} - Original', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Value', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(df_for_scaling[feature2], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[0, 1].set_title(f'{feature2} - Original', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Value', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# StandardScaler
axes[1, 0].hist(df_standard_scaled[feature1], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[1, 0].set_title(f'{feature1} - StandardScaler', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Value', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(df_standard_scaled[feature2], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[1, 1].set_title(f'{feature2} - StandardScaler', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Value', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

# MinMaxScaler
axes[2, 0].hist(df_minmax_scaled[feature1], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[2, 0].set_title(f'{feature1} - MinMaxScaler', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Value', fontsize=10)
axes[2, 0].grid(True, alpha=0.3)

axes[2, 1].hist(df_minmax_scaled[feature2], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[2, 1].set_title(f'{feature2} - MinMaxScaler', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Value', fontsize=10)
axes[2, 1].grid(True, alpha=0.3)

# RobustScaler
axes[3, 0].hist(df_robust_scaled[feature1], bins=50, color='blue', alpha=0.7, edgecolor='black')
axes[3, 0].set_title(f'{feature1} - RobustScaler', fontsize=12, fontweight='bold')
axes[3, 0].set_xlabel('Value', fontsize=10)
axes[3, 0].grid(True, alpha=0.3)

axes[3, 1].hist(df_robust_scaled[feature2], bins=50, color='green', alpha=0.7, edgecolor='black')
axes[3, 1].set_title(f'{feature2} - RobustScaler', fontsize=12, fontweight='bold')
axes[3, 1].set_xlabel('Value', fontsize=10)
axes[3, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Scaling comparison complete")

## Section 9: Feature Selection

Identify the most important features for predicting load.

In [ ]:
# Prepare data for feature selection
# We'll predict load_mw using other features

# Select features (exclude target, timestamp, and features derived from target)
feature_cols = ['voltage_kv', 'current_a', 'frequency_hz', 'power_factor',
                'temperature_c', 'wind_speed_ms', 'solar_irradiance', 'humidity_percent',
                'hour', 'day_of_week', 'month', 'is_weekend', 'is_business_hours',
                'is_peak_hours', 'hour_sin', 'hour_cos', 'day_of_week_sin', 'day_of_week_cos']

# Create subset with complete cases (no NaN values)
# Start from row 200 to avoid NaN from lag features
df_complete = df.iloc[200:][feature_cols + ['load_mw']].dropna().copy()

# Split into features (X) and target (y)
X = df_complete[feature_cols]
y = df_complete['load_mw']

print(f"Feature selection dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Target variable: load_mw")
print(f"\nFeatures:\n{feature_cols}")

In [ ]:
# Method 1: Variance Threshold
# Remove features with low variance (features that don't change much)
# Low variance features provide little information

# Calculate variance for each feature
variances = X.var()
print("Feature Variances:")
print(variances.sort_values(ascending=False))

# Apply variance threshold (remove features with variance < 0.01)
var_threshold = VarianceThreshold(threshold=0.01)
X_high_var = var_threshold.fit_transform(X)

# Get selected feature names
selected_features = X.columns[var_threshold.get_support()].tolist()

print(f"\nFeatures after variance threshold: {len(selected_features)}")
print(f"Removed features: {set(feature_cols) - set(selected_features)}")

In [ ]:
# Method 2: Univariate Feature Selection (F-statistic)
# Select features based on statistical tests
# F-statistic measures linear relationship between each feature and target

# Select top 10 features using F-statistic
selector_f = SelectKBest(score_func=f_regression, k=10)
selector_f.fit(X, y)

# Get feature scores
feature_scores_f = pd.DataFrame({
    'Feature': feature_cols,
    'F_Score': selector_f.scores_,
    'P_Value': selector_f.pvalues_
}).sort_values('F_Score', ascending=False)

print("Top 10 Features by F-Statistic:")
print(feature_scores_f.head(10))

# Visualize feature scores
plt.figure(figsize=(14, 8))
plt.barh(range(len(feature_scores_f)), feature_scores_f['F_Score'], color='steelblue', edgecolor='black')
plt.yticks(range(len(feature_scores_f)), feature_scores_f['Feature'])
plt.xlabel('F-Score', fontsize=12, fontweight='bold')
plt.title('Feature Importance: F-Statistic Scores', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Method 3: Mutual Information
# Measures both linear and non-linear relationships
# More comprehensive than F-statistic

# Select top 10 features using mutual information
selector_mi = SelectKBest(score_func=mutual_info_regression, k=10)
selector_mi.fit(X, y)

# Get feature scores
feature_scores_mi = pd.DataFrame({
    'Feature': feature_cols,
    'MI_Score': selector_mi.scores_
}).sort_values('MI_Score', ascending=False)

print("Top 10 Features by Mutual Information:")
print(feature_scores_mi.head(10))

# Visualize feature scores
plt.figure(figsize=(14, 8))
plt.barh(range(len(feature_scores_mi)), feature_scores_mi['MI_Score'], color='coral', edgecolor='black')
plt.yticks(range(len(feature_scores_mi)), feature_scores_mi['Feature'])
plt.xlabel('Mutual Information Score', fontsize=12, fontweight='bold')
plt.title('Feature Importance: Mutual Information Scores', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Compare feature selection methods
comparison_df = pd.DataFrame({
    'Feature': feature_cols,
    'F_Score': selector_f.scores_,
    'MI_Score': selector_mi.scores_
})

# Normalize scores to 0-1 for comparison
comparison_df['F_Score_Norm'] = (comparison_df['F_Score'] - comparison_df['F_Score'].min()) / (comparison_df['F_Score'].max() - comparison_df['F_Score'].min())
comparison_df['MI_Score_Norm'] = (comparison_df['MI_Score'] - comparison_df['MI_Score'].min()) / (comparison_df['MI_Score'].max() - comparison_df['MI_Score'].min())

# Calculate average score
comparison_df['Avg_Score'] = (comparison_df['F_Score_Norm'] + comparison_df['MI_Score_Norm']) / 2
comparison_df = comparison_df.sort_values('Avg_Score', ascending=False)

print("Feature Selection Comparison (Normalized Scores):")
print(comparison_df[['Feature', 'F_Score_Norm', 'MI_Score_Norm', 'Avg_Score']].head(15))

# Visualize comparison
fig, ax = plt.subplots(figsize=(14, 10))

x = np.arange(len(comparison_df))
width = 0.35

ax.barh(x - width/2, comparison_df['F_Score_Norm'], width, label='F-Statistic', color='steelblue', edgecolor='black')
ax.barh(x + width/2, comparison_df['MI_Score_Norm'], width, label='Mutual Information', color='coral', edgecolor='black')

ax.set_yticks(x)
ax.set_yticklabels(comparison_df['Feature'])
ax.set_xlabel('Normalized Score', fontsize=12, fontweight='bold')
ax.set_title('Feature Selection Methods Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nTop 10 Features (Combined Ranking):")
print(comparison_df['Feature'].head(10).tolist())

## Section 10: Final Feature Summary and Export

In [ ]:
# Summary of all engineered features
print("=" * 80)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 80)

# Count features by category
original_features = 14
temporal_features = len([col for col in df.columns if any(x in col for x in ['hour', 'day', 'month', 'week', 'quarter', 'is_weekend', 'is_business', 'is_peak', 'is_night', 'is_summer', 'is_winter'])])
lag_features = len([col for col in df.columns if 'lag' in col or 'rolling' in col or 'ewma' in col or 'cumulative' in col])
electrical_features = len([col for col in df.columns if any(x in col for x in ['deviation', 'ratio', 'impedance', 'factor', 'thd', 'transformer', 'oil', 'renewable', 'wind_power', 'solar_capacity'])])
interaction_features = len([col for col in df.columns if 'interaction' in col or 'squared' in col or 'poly' in col])
change_features = len([col for col in df.columns if 'change' in col or 'pct_change' in col])

print(f"\n1. ORIGINAL FEATURES: {original_features}")
print(f"   - Raw sensor measurements and weather data")

print(f"\n2. TEMPORAL FEATURES: {temporal_features}")
print(f"   - Hour, day, month, cyclical encodings, binary time flags")

print(f"\n3. LAG & ROLLING FEATURES: {lag_features}")
print(f"   - Historical values, rolling statistics, EWMA")

print(f"\n4. ELECTRICAL DOMAIN FEATURES: {electrical_features}")
print(f"   - Voltage/frequency deviations, power factors, transformer metrics")

print(f"\n5. INTERACTION & POLYNOMIAL FEATURES: {interaction_features}")
print(f"   - Feature combinations, squared terms, polynomial expansions")

print(f"\n6. RATE OF CHANGE FEATURES: {change_features}")
print(f"   - Hour-over-hour, day-over-day changes, acceleration")

print(f"\n{'='*80}")
print(f"TOTAL FEATURES: {len(df.columns)}")
print(f"ORIGINAL → ENGINEERED: {original_features} → {len(df.columns)} ({len(df.columns)/original_features:.1f}x increase)")
print(f"{'='*80}")

print(f"\nDataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Display sample of engineered features
print("Sample of Engineered Dataset:")
print(df.iloc[200:205].T)  # Transpose for better readability

In [ ]:
# Save engineered features dataset
output_path = '../datasets/power_system_data_engineered.csv'
df.to_csv(output_path, index=False)

print(f"Engineered dataset saved to: {output_path}")
print(f"File size: {len(df)} rows × {len(df.columns)} columns")
print(f"\nThis dataset is ready for machine learning model training!")

## What This Means for Electrical Engineers

### Industry Relevance:

1. **Predictive Maintenance**: Engineered features like oil temperature rise, loading margins, and rate-of-change metrics are crucial for predicting equipment failures before they occur. Utilities save millions by avoiding unplanned outages.

2. **Load Forecasting**: Temporal features (cyclical encodings, lag features, rolling statistics) are essential for accurate load forecasting. Forecasting errors of 1-2% vs 5-10% can mean millions in energy market operations.

3. **Renewable Integration**: Features like wind power potential, solar capacity factor, and combined renewable potential help optimize dispatch of renewable resources and manage grid stability.

4. **Power Quality Monitoring**: Voltage/frequency deviation features, THD estimation, and power factor metrics are critical for regulatory compliance and customer service.

5. **Real-Time Operations**: Rate-of-change features (ramping rates) are monitored in real-time control centers to ensure generation can track load changes and maintain frequency.

### Key Takeaways:

- **Domain knowledge drives feature engineering**: Understanding electrical systems helps create meaningful features
- **Cyclical encoding matters**: Time features must be encoded cyclically for ML models to understand temporal proximity
- **Multiple time scales are important**: Hourly, daily, weekly, and seasonal patterns all impact power systems
- **Lag features capture dependencies**: Time series forecasting requires historical context
- **Scaling is algorithm-dependent**: Choose scaling method based on your ML algorithm and data characteristics
- **More features ≠ better models**: Feature selection prevents overfitting and improves interpretability
- **Physical relationships matter**: Features derived from electrical formulas (P, Q, S relationships) improve model physics consistency

### Common Mistakes:

- **Linear encoding of cyclical features**: Using hour as 0-23 instead of sin/cos
- **Look-ahead bias**: Using future information in lag features (e.g., shift(-1) instead of shift(1))
- **Ignoring domain constraints**: Creating features that violate physical laws
- **Forgetting to scale**: Many ML algorithms require scaled features
- **Over-engineering**: Creating hundreds of features without selection leads to overfitting
- **Not handling NaN values**: Lag/rolling features create missing values that must be handled

### Pro Tips:

- Start simple, add complexity gradually
- Visualize engineered features to understand their behavior
- Document feature engineering pipeline for reproducibility
- Use feature selection to identify most important features
- Consider computational cost of complex features for real-time applications
- Validate that engineered features make physical sense
- Save scaler parameters for consistent transform in production
- Test different scaling methods for your specific problem

### Next Steps:

In Module 2, we'll use these engineered features to build machine learning models. You'll see how good feature engineering directly translates to better model performance, faster training, and more interpretable results.

The features we've created here are based on real-world practices in power system operations and will serve as the foundation for all subsequent machine learning tasks in this course.